<a href="https://colab.research.google.com/github/MiguelFigueroa02/ITAM_Sport_Lucho_FC/blob/main/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Exploración archivos Json
---
A partir de los archivos json disponibles, procesamos los datos para construir un dataset que contenga todos los partidos del américa disponibles en el periodo 2023 - 2026

In [10]:
import json
import pandas as pd
from pathlib import Path

In [11]:
DATA_DIR = Path("/content/")
OUTPUT_FILE = Path("/content/america_matches.csv")
AMERICA_ID = 1229

In [12]:
json_files = sorted(DATA_DIR.glob("matches-*.json"))

In [13]:
if not json_files:
    raise FileNotFoundError(
        f"No se encontraron archivos JSON en: {DATA_DIR.resolve()}"
    )

In [16]:
records = []

for file in json_files:

    print(f"\nArchivo: {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        matches = json.load(f)

    print(f"Partidos encontrados en archivo: {len(matches)}")

    for match in matches:

        home = match["home_team"]
        away = match["away_team"]

        home_id = home["home_team_id"]
        away_id = away["away_team_id"]

        if home_id != AMERICA_ID and away_id != AMERICA_ID:
            continue

        is_home = home_id == AMERICA_ID

        opponent = (
            away["away_team_name"]
            if is_home
            else home["home_team_name"]
        )

        america_score = (
            match["home_score"]
            if is_home
            else match["away_score"]
        )

        opponent_score = (
            match["away_score"]
            if is_home
            else match["home_score"]
        )


        managers = home.get("managers", []) if is_home else away.get("managers", [])

        coach = (
            managers[0]["name"]
            if managers
            else None
        )


        records.append({

            "match_id": match["match_id"],

            "match_date": match["match_date"],

            "kick_off": match.get("kick_off"),

            "season": match["season"]["season_name"],

            "competition": match["competition"]["competition_name"],

            "competition_stage": (
                match.get("competition_stage", {}).get("name")
            ),

            "match_week": match.get("match_week"),

            "opponent": opponent,

            "venue": "Home" if is_home else "Away",

            "america_score": america_score,

            "opponent_score": opponent_score,

            "coach": coach,

            "stadium": (
                match.get("stadium", {}).get("name")
            ),

            "match_status": match.get("match_status"),

            "match_status_360": match.get("match_status_360"),

        })



america_matches = pd.DataFrame(records)


if america_matches.empty:
    raise ValueError(
        "No se encontraron partidos del América. "
        "Revisa el AMERICA_ID."
    )


def get_result(row):

    if row["america_score"] > row["opponent_score"]:
        return "W"

    elif row["america_score"] == row["opponent_score"]:
        return "D"

    else:
        return "L"


america_matches["result"] = america_matches.apply(
    get_result,
    axis=1
)


america_matches["match_date"] = pd.to_datetime(
    america_matches["match_date"]
)


america_matches = america_matches.sort_values(
    ["match_date", "match_id"]
).reset_index(drop=True)



OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)



america_matches.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 60)
print("CATÁLOGO DEL AMÉRICA")
print("=" * 60)

print(f"\nTotal de partidos: {len(america_matches)}")

print("\nPartidos por temporada:")
print(
    america_matches
    .groupby("season")
    .size()
    .sort_index()
)

print("\nResultados:")
print(
    america_matches["result"]
    .value_counts()
)

print("\nLocal / Visitante:")
print(
    america_matches["venue"]
    .value_counts()
)

print("\nDisponibilidad de 360:")
print(
    america_matches["match_status_360"]
    .value_counts(dropna=False)
)

print("\nEntrenadores:")
print(
    america_matches["coach"]
    .value_counts(dropna=False)
)

print(f"\nArchivo generado:")
print(OUTPUT_FILE.resolve())

print("\nPrimeros registros:")
print(
    america_matches.head(10).to_string(index=False)
)


Archivo: matches-2023-2024.json
Partidos encontrados en archivo: 340

Archivo: matches-2024-2025.json
Partidos encontrados en archivo: 340

Archivo: matches-2025-2026.json
Partidos encontrados en archivo: 337

CATÁLOGO DEL AMÉRICA

Total de partidos: 131

Partidos por temporada:
season
2023/2024    46
2024/2025    47
2025/2026    38
dtype: int64

Resultados:
result
W    69
D    37
L    25
Name: count, dtype: int64

Local / Visitante:
venue
Away    66
Home    65
Name: count, dtype: int64

Disponibilidad de 360:
match_status_360
available      130
unscheduled      1
Name: count, dtype: int64

Entrenadores:
coach
André Soares Jardine              129
Diego Alberto Cervantes Chávez      2
Name: count, dtype: int64

Archivo generado:
/content/america_matches.csv

Primeros registros:
 match_id match_date     kick_off    season competition competition_stage  match_week    opponent venue  america_score  opponent_score                coach                stadium match_status match_status_360 r

## 2.  Auditoría inicial de los datos de los eventos de partidos.
---

In [42]:
def evaluacion(archivo):
  repo = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/raw/'
  path = repo + archivo
  CHUNK_SIZE = 50_000
  print("### Evaluación de eventos")
  header = pd.read_csv(
      path,
      nrows=0
    )
  columns = header.columns.tolist()
  print(f"\nNúmero de columnas: {len(columns)}")
  print("\nColumnas:")
  for i, col in enumerate(columns, start=1):
      print(f"{i:3}. {col}")

  total_rows = 0
  unique_matches = set()

  sample = None

  for chunk in pd.read_csv(
      path,
      chunksize=CHUNK_SIZE,
      low_memory=False,
      on_bad_lines='skip'
  ):

      total_rows += len(chunk)

      if sample is None:
          sample = chunk.head(5).copy()

      if "match_id" in chunk.columns:
          unique_matches.update(
              chunk["match_id"].dropna().unique()
          )

  print("### RESUMEN")

  print(f"\nFilas totales: {total_rows:,}")

  print(f"Partidos únicos: {len(unique_matches):,}")

  print("### MUESTRA DE EVENTOS")

  print(
    sample.to_csv(index=False)
  )
  print("### TIPOS DE DATOS")

  print(sample.dtypes)


In [43]:
evaluacion('Liga_MX_2023_2024_America_p1.csv')

### Evaluación de eventos

Número de columnas: 251

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [44]:
evaluacion('Liga_MX_2023_2024_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [45]:
evaluacion('Liga_MX_2023_2024_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [46]:
evaluacion('Liga_MX_2024_2025_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [47]:
evaluacion('Liga_MX_2024_2025_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [48]:
evaluacion('Liga_MX_2024_2025_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [49]:
evaluacion('Liga_MX_2025_2026_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [50]:
evaluacion('Liga_MX_2025_2026_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a